# LC 33 — Search in Rotated Sorted Array
**Difficulty:** Medium | **Category:** Binary Search
**Pattern:** Binary Search on Rotated Array (Search)

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> One of the two halves around mid is
always fully sorted. Determine which half is sorted, check if the
target falls within that sorted range, then commit to that half or
the other. Two decisions per step keeps it O(log n).
</div>

## Official Problem Statement

There is an integer array `nums` sorted in ascending order (with
**unique** values) that has been possibly rotated at an unknown pivot
index `k`.

Given the array `nums` after the possible rotation and an integer
`target`, return the index of `target` if it is in `nums`, or `-1`
if it is not in `nums`.

You must write an algorithm with `O(log n)` runtime complexity.

**Constraints:**
- `1 <= nums.length <= 5000`
- `-10^4 <= nums[i] <= 10^4`
- All values of `nums` are **unique**.
- `nums` is an ascending array that is possibly rotated.
- `-10^4 <= target <= 10^4`

## What This Is Actually Asking

You are searching for a value in a sorted list that has been cut and
swapped — similar to LC 153 but now you want a specific target, not
just the minimum. Normal binary search breaks because the array is not
monotonically increasing end-to-end. The trick is that no matter where
you cut, at least one of the two halves will always be cleanly sorted,
and you can use that guarantee to decide which side to search.

## Walk Through an Example by Hand

```
nums = [4, 5, 6, 7, 0, 1, 2],  target = 0

Step 1: lo=0, hi=6, mid=3  nums[mid]=7
        Left sorted? nums[0]=4 <= nums[3]=7  YES
        Is 0 in [4..7)?  No  => go RIGHT  => lo=4

Step 2: lo=4, hi=6, mid=5  nums[mid]=1
        Left sorted? nums[4]=0 <= nums[5]=1  YES
        Is 0 in [0..1)?  Yes => go LEFT  => hi=4

Step 3: lo=4, hi=4, mid=4  nums[mid]=0 == 0  => return 4
```

## The Picture

```
nums = [4,  5,  6,  7,  0,  1,  2],  target = 0
idx  =  0   1   2   3   4   5   6

Round 1:
  lo=0               hi=6
  [4,  5,  6,  7,  0,  1,  2]
               ^mid=3
  nums[0]=4 <= nums[3]=7  => LEFT half [0..3] is sorted
  target=0 NOT in [4,7)   => search RIGHT  => lo=4

Round 2:
              lo=4      hi=6
  [ _,  _,  _,  _,  0,  1,  2]
                          ^mid=5
  nums[4]=0 <= nums[5]=1  => LEFT half [4..5] is sorted
  target=0 IS in [0,1)    => search LEFT   => hi=4

Round 3:
              lo=hi=4
  [ _,  _,  _,  _,  0,  _,  _]
                      ^mid=4
  nums[4]=0 == target=0   => return 4  FOUND
```

## When To Use This Pattern

- When the array is **rotated sorted** and you need to find a specific
  value, not just the minimum.
- When `nums[lo] <= nums[mid]` is the key test — it tells you the
  left half is sorted regardless of where the pivot is.
- When you need `O(log n)` on a once-rotated array with unique values.
- When you see both "sorted" and "rotated" in the same problem — this
  pattern handles the combination directly.
- After mastering LC 153 (find minimum) — this is the natural next step.

## The Approach

Use standard binary search pointers `lo` and `hi`. At each `mid`,
first check for a direct match. Then determine which half is cleanly
sorted: if `nums[lo] <= nums[mid]`, the left half is sorted. Check
whether the target lies strictly within that sorted range; if yes,
narrow to the left, otherwise narrow to the right. Apply the mirror
logic when the right half is sorted. Return `-1` if the window closes
without a match.

In [1]:
from typing import List  # standard type hints

In [2]:
def test_harness(func):
    """Run test cases for LC 33 Search in Rotated Sorted Array."""
    tests = [
        # (nums, target, expected)
        ([4, 5, 6, 7, 0, 1, 2], 0,  4),  # target in right portion
        ([4, 5, 6, 7, 0, 1, 2], 3, -1),  # target not present
        ([1],                   0, -1),  # single element miss
        ([1],                   1,  0),  # single element hit
        ([3, 1],                1,  1),  # two elements rotated
        ([5, 1, 3],             5,  0),  # target is first element
        ([4, 5, 6, 7, 0, 1, 2], 4,  0),  # target is lo element
    ]
    passed = 0
    for nums, target, expected in tests:
        result = func(nums, target)
        status = "PASSED" if result == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        else:
            print(
                f"{status} | nums={nums} target={target} "
                f"| got={result} expected={expected}"
            )
    print(f"\n{passed}/{len(tests)} tests passed.")

In [9]:
def search(nums: List[int], target: int) -> int:
    """
    Search for target in a rotated sorted array.

    Args:
        nums:   Rotated sorted array of unique integers.
        target: Value to find.

    Returns:
        Index of target if found, else -1.

    Time:  O(log n)
    Space: O(1)
    """
    l, r = 0, len(nums) - 1
    
    while l <= r:
        mid = (l + r) // 2
    
        if nums[mid] == target:
            return mid
    
        # left half is the clean sorted side
        if nums[l] <= nums[mid]:
            # target is either to the right of mid, or left of the pivot — go right
            # target > mid or because left is the upper sorted portion
            # and target  less than this portion
            if target > nums[mid] or target < nums[l]:
                l = mid + 1
            else:
                # target is somewhere in the clean left half — go left
                r = mid - 1
    
        # right half is the clean sorted side
        else:
            # target < mid or because right is the lower sorted portion
            # and target  bigger than this portion
            if target < nums[mid] or target > nums[r]:
                r = mid - 1
            else:
                # target is somewhere in the clean right half — go right
                l = mid + 1
    return -1  
   


# --- Debug prints (remove before final submission) ---
# Expected: 4
print(search([4, 5, 6, 7, 0, 1, 2], 0))

# Expected: -1
print(search([4, 5, 6, 7, 0, 1, 2], 3))

# Expected: 0  (single element hit)
print(search([1], 1))

# Expected: 1  (two elements, target at index 1)
print(search([3, 1], 1))

# Expected: 0  (target is lo)
print(search([4, 5, 6, 7, 0, 1, 2], 4))
test_harness(search)

4
-1
0
1
0

7/7 tests passed.


In [ ]:
# Uncomment and run when solution is ready
# test_harness(search)

## Complexity

| Approach                | Time     | Space |
|-------------------------|----------|-------|
| Brute Force (linear)    | O(n)     | O(1)  |
| Binary Search (optimal) | O(log n) | O(1)  |

## Real World Connection

At Citi, version-controlled config records are sometimes stored in
ring buffers that wrap around — effectively a rotated sorted structure
ordered by version number. Searching for a specific config version
across 6,000 endpoints without O(log n) lookup would slow deployment
pipelines. AWS Systems Manager Parameter Store uses similar binary
search strategies on internally segmented sorted key spaces to resolve
parameter lookups in milliseconds even as parameter counts scale into
the tens of thousands.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra